# Cloud Fact-Checker — GPU retraining

Standalone Colab workflow for the repository's DistilBERT v2 model. It preserves the production label contract (`0 = Misinformation`, `1 = Truth`) and the deterministic 70/15/15 ISOT split used by `train_model_v2.py`.

Before running, choose **Runtime → Change runtime type → T4 GPU**. Run cells from top to bottom. Checkpoints, the final model, and JSON reports are written to Google Drive, so a disconnected runtime can resume. This notebook does not connect to or change the local Docker deployment.

In [ ]:
# Verify that Colab actually assigned a CUDA GPU.
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select Runtime > Change runtime type > T4 GPU, then reconnect.")
subprocess.run(["nvidia-smi"], check=True)
print(f"PyTorch {torch.__version__}; GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Install the same pinned ML stack used by the repository.
%pip install -q transformers==4.47.1 datasets==3.2.0 accelerate==1.2.1 scikit-learn==1.6.0 pandas==2.2.3

## Storage configuration

The default expects `Fake.csv` and `True.csv` in `MyDrive/Cloud_Project/data`. Set `DATA_SOURCE = "upload"` to choose both CSVs from your computer instead. Uploaded data is copied into Drive before training.

To record baseline metrics, place the current model directory at `MyDrive/Cloud_Project/models/misinformation_model`, or change `BASELINE_MODEL_DIR`. If it is absent, baseline evaluation is skipped without blocking v2 training.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/Cloud_Project")
DATA_DIR = PROJECT_DIR / "data"
RUN_DIR = PROJECT_DIR / "training_v2"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
FINAL_MODEL_DIR = RUN_DIR / "misinformation_model_v2"
REPORT_DIR = RUN_DIR / "reports"
BASELINE_MODEL_DIR = PROJECT_DIR / "models" / "misinformation_model"
DATA_SOURCE = "drive"  # Change to "upload" to select CSVs interactively.

for path in (DATA_DIR, CHECKPOINT_DIR, FINAL_MODEL_DIR, REPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)

if DATA_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    for required in ("Fake.csv", "True.csv"):
        matches = [name for name in uploaded if Path(name).name.lower() == required.lower()]
        if not matches:
            raise FileNotFoundError(f"Upload a file named {required}")
        (DATA_DIR / required).write_bytes(uploaded[matches[0]])
elif DATA_SOURCE != "drive":
    raise ValueError("DATA_SOURCE must be 'drive' or 'upload'")

FAKE_CSV = DATA_DIR / "Fake.csv"
TRUE_CSV = DATA_DIR / "True.csv"
for path in (FAKE_CSV, TRUE_CSV):
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Upload it or correct DATA_DIR.")
print(f"Persistent run directory: {RUN_DIR}")

In [ ]:
# Imports, deterministic settings, and the repository's model constants.
import json
import random
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from transformers import (
    DataCollatorWithPadding, DistilBertForSequenceClassification,
    DistilBertTokenizerFast, EarlyStoppingCallback, Trainer,
    TrainingArguments, set_seed,
)
from transformers.trainer_utils import get_last_checkpoint

SEED = 42
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
NUM_EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 1e-5
LABELS = {0: "Misinformation", 1: "Truth"}
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

In [ ]:
# Load and split exactly like backend/app/train_model_v2.py.
def load_isot(fake_path, true_path):
    fake = pd.read_csv(fake_path)
    true = pd.read_csv(true_path)
    required = {"title", "text"}
    for name, frame in (("Fake.csv", fake), ("True.csv", true)):
        missing = required - set(frame.columns)
        if missing:
            raise ValueError(f"{name} is missing columns: {sorted(missing)}")
    fake["text"] = (fake["title"].fillna("") + " " + fake["text"].fillna("")).str.strip()
    true["text"] = (true["title"].fillna("") + " " + true["text"].fillna("")).str.strip()
    fake["label"] = 0
    true["label"] = 1
    fake = fake[["text", "label"]].dropna(subset=["text"])
    true = true[["text", "label"]].dropna(subset=["text"])
    fake = fake[fake["text"].str.split().str.len() >= 10]
    true = true[true["text"].str.split().str.len() >= 10]
    return pd.concat([fake, true], ignore_index=True)

def deterministic_split(data):
    train, holdout = train_test_split(
        data, test_size=0.30, random_state=SEED, stratify=data["label"]
    )
    validation, test = train_test_split(
        holdout, test_size=0.50, random_state=SEED, stratify=holdout["label"]
    )
    return tuple(part.reset_index(drop=True) for part in (train, validation, test))

data = load_isot(FAKE_CSV, TRUE_CSV)
train_df, validation_df, test_df = deterministic_split(data)
print({"all": len(data), "train": len(train_df), "validation": len(validation_df), "test": len(test_df)})
print("Class counts:", data["label"].value_counts().sort_index().to_dict())

In [ ]:
# Shared evaluation: baseline and v2 always use the same held-out test_df.
def metric_values(labels, predictions):
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "f1": float(f1_score(labels, predictions, zero_division=0)),
        "test_size": int(len(labels)),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }

def evaluate_checkpoint(model_dir, report_name):
    tokenizer = DistilBertTokenizerFast.from_pretrained(model_dir)
    model = DistilBertForSequenceClassification.from_pretrained(model_dir).cuda().eval()
    predictions, confidences = [], []
    for start in range(0, len(test_df), 64):
        batch = test_df["text"].iloc[start:start + 64].tolist()
        encoded = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH)
        encoded = {key: value.cuda() for key, value in encoded.items()}
        with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
            probabilities = torch.softmax(model(**encoded).logits, dim=-1)
        predictions.extend(probabilities.argmax(dim=-1).cpu().tolist())
        confidences.extend(probabilities.max(dim=-1).values.cpu().tolist())
    report = metric_values(test_df["label"].tolist(), predictions)
    report.update({
        "model_path": str(model_dir),
        "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
        "label_contract": {"0": "Misinformation", "1": "Truth"},
        "average_confidence": float(np.mean(confidences)),
    })
    report_path = REPORT_DIR / report_name
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    del model
    torch.cuda.empty_cache()
    print(json.dumps(report, indent=2))
    print(f"Saved {report_path}")
    return report

baseline_report = None
if (BASELINE_MODEL_DIR / "config.json").is_file():
    baseline_report = evaluate_checkpoint(BASELINE_MODEL_DIR, "baseline_metrics.json")
else:
    print(f"Baseline skipped: no checkpoint found at {BASELINE_MODEL_DIR}")

In [ ]:
# Tokenize once. The label metadata is embedded in every saved checkpoint.
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def as_dataset(frame):
    dataset = Dataset.from_dict({"text": frame["text"].tolist(), "label": frame["label"].tolist()})
    return dataset.map(
        lambda batch: tokenizer(batch["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH),
        batched=True, remove_columns=["text"]
    )

train_dataset = as_dataset(train_df)
validation_dataset = as_dataset(validation_df)
test_dataset = as_dataset(test_df)

def trainer_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "f1": float(f1_score(labels, predictions, zero_division=0)),
    }

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=LABELS,
    label2id={"Misinformation": 0, "Truth": 1},
)

In [ ]:
# Train on GPU. Re-running this cell resumes the newest Drive checkpoint.
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    overwrite_output_dir=False,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,
    seed=SEED,
    data_seed=SEED,
    report_to=[],
)
trainer = Trainer(
    model=model, args=training_args, train_dataset=train_dataset,
    eval_dataset=validation_dataset, compute_metrics=trainer_metrics,
    data_collator=DataCollatorWithPadding(tokenizer),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.0)],
)
last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))
print("Resuming from:", last_checkpoint or "fresh pretrained model")
train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))
run_metadata = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "base_model": MODEL_NAME, "seed": SEED, "max_length": MAX_LENGTH,
    "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
    "split": {"train": len(train_df), "validation": len(validation_df), "test": len(test_df)},
    "label_contract": {"0": "Misinformation", "1": "Truth"},
    "train_metrics": train_result.metrics,
}
(REPORT_DIR / "training_metadata.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
print(f"Final model saved persistently to {FINAL_MODEL_DIR}")

In [ ]:
# Final held-out report and optional baseline comparison.
v2_report = evaluate_checkpoint(FINAL_MODEL_DIR, "v2_metrics.json")
if baseline_report:
    comparison = {
        metric: {"baseline": baseline_report[metric], "v2": v2_report[metric],
                 "delta": v2_report[metric] - baseline_report[metric]}
        for metric in ("accuracy", "precision", "recall", "f1")
    }
    (REPORT_DIR / "baseline_vs_v2.json").write_text(json.dumps(comparison, indent=2), encoding="utf-8")
    print(json.dumps(comparison, indent=2))

In [ ]:
# Claim smoke tests. These are diagnostic, not evidence-grounded fact-checking scores.
smoke_claims = [
    {"text": "The Earth orbits the Sun.", "expected": "Truth"},
    {"text": "The Earth is flat.", "expected": "Misinformation"},
    {"text": "Water freezes at zero degrees Celsius at standard pressure.", "expected": "Truth"},
    {"text": "Humans can breathe normally in outer space without equipment.", "expected": "Misinformation"},
]
smoke_tokenizer = DistilBertTokenizerFast.from_pretrained(FINAL_MODEL_DIR)
smoke_model = DistilBertForSequenceClassification.from_pretrained(FINAL_MODEL_DIR).cuda().eval()
smoke_results = []
for item in smoke_claims:
    encoded = smoke_tokenizer(item["text"], return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    encoded = {key: value.cuda() for key, value in encoded.items()}
    with torch.inference_mode():
        probabilities = torch.softmax(smoke_model(**encoded).logits, dim=-1)[0]
    class_id = int(probabilities.argmax())
    smoke_results.append({**item, "predicted": LABELS[class_id],
                          "confidence_percent": round(float(probabilities[class_id]) * 100, 2),
                          "matches_expected": LABELS[class_id] == item["expected"]})
smoke_path = REPORT_DIR / "claim_smoke_tests.json"
smoke_path.write_text(json.dumps(smoke_results, indent=2), encoding="utf-8")
print(json.dumps(smoke_results, indent=2))
print("Note: ISOT is an article-style dataset; passing these claims does not make the model a general fact checker.")

## Export

The model already persists in Drive at `MyDrive/Cloud_Project/training_v2/misinformation_model_v2`. Run the next cell to make a deployment ZIP and download it. Keep the Drive copy until local evaluation and deployment are complete.

In [ ]:
from google.colab import files

archive_base = RUN_DIR / "misinformation_model_v2"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=FINAL_MODEL_DIR.parent, base_dir=FINAL_MODEL_DIR.name))
print(f"ZIP saved in Drive: {archive_path} ({archive_path.stat().st_size / 1024**2:.1f} MiB)")
# The browser download can take time; comment this line out if the Drive copy is enough.
files.download(str(archive_path))